# Visualization of simulation

The code below visualises the simulation that replicates the environment and bee
behavior of research paper "Simulation-governed design and tuning of greenhouses 
for successful bee pollination".

The code replicates specifically the plant layout "MFMFMFMF" only. The detailed
environment settings can be found in the research paper above. There are two major
differences:
    1. The implemented environment is continuous not grid
    2. Origianl grid environment has cells 0.35m x 0.35m. This seems to be a typo
       or error (from last time talking with the first author, should of been 
       0.7m x 0.7m). If overlapping the grid and continuous environment, plants
       from the continuous environment will land in the middle of the cells from
       the grid environment (where the cell represents a plant).

In the paper, Levy flight was simulated using Cauchy Distribution. I did not find
the exact parameters used for the Cauchy Distribution, so I used the standard
values. Otherwise, all plant and bee behavior should replicate the behaviors defined
in the research paper.

Following notes are for the visualization where the number of bees 
- Currently just simulates the scenario where the layouts are "MFMFMFMF".
- Red represents female flowers
- Blue represents male flowers
- Red/Blue with white boarders represents the flower has been visited once, and the bee will not land on it again
- Orange represents pollinated female flowers
- Black represents bees with no pollen
- White with blackboarder represents bees with pollen

In [ ]:
%matplotlib notebook

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np
import pickle
import pandas as pd
from tqdm import tqdm
import matplotlib as mpl

def visualize_simulation(dynamic_plants=True):
    """
    Input:
       - dynamic_plants (bool): If True, plants are dynamic in the visualization.
                                Otherwise, they are static. Note when set to true
                                it may take longer to render the animation and use
                                up more memory.
    
    """

    # Increase animation embed size limit
    mpl.rcParams['animation.embed_limit'] = 50

    # --------------------------
    # Load simulation data
    # --------------------------
    with open("simulation_data.pkl", "rb") as f:
        df = pickle.load(f)

    bee_df = df[df["Agent Type"] == "bee"]
    plant_df = df[df["Agent Type"] == "plant"]
    static_plant_df = plant_df[plant_df.index.get_level_values("Step") == 0]
    steps = sorted(bee_df.index.get_level_values("Step").unique())

    # --------------------------
    # Plot setup
    # --------------------------
    fig, ax = plt.subplots(figsize=(72/4, 18/4))
    ax.set_facecolor("gray")

    x_min, x_max = 0, 72 * 0.7
    y_min, y_max = 0, 18 * 0.7
    ax.plot([x_min, x_max], [y_min, y_min], color="black", linewidth=1.5)
    ax.plot([x_min, x_max], [y_max, y_max], color="black", linewidth=1.5)
    ax.plot([x_min, x_min], [y_min, y_max], color="black", linewidth=1.5)
    ax.plot([x_max, x_max], [y_min, y_max], color="black", linewidth=1.5)

    all_bee_positions = bee_df["Bee Positions"].tolist()
    all_plant_positions = plant_df["Bee Positions"].tolist()
    all_x = [p[0] for p in all_bee_positions + all_plant_positions]
    all_y = [p[1] for p in all_bee_positions + all_plant_positions]
    ax.set_xlim(min(all_x) - 1, max(all_x) + 1)
    ax.set_ylim(min(all_y) - 1, max(all_y) + 1)
    ax.set_xlabel("X Position")
    ax.set_ylabel("Y Position")
    ax.set_title("Bee Movement Relative to Plants")
    ax.set_xlim(x_min - 2, x_max + 2)
    ax.set_ylim(y_min - 2, y_max + 2)

    # --------------------------
    # Plant: static or dynamic
    # --------------------------
    if not dynamic_plants:
        for _, plant in static_plant_df.iterrows():
            pos = plant["Bee Positions"]
            style = plant["Plot Code"]
            assert isinstance(style, dict), f"Invalid Plot Code in plant: {style}"
            ax.scatter(pos[0], pos[1],
                       marker=style.get("marker", "o"),
                       c=style.get("color"),
                       edgecolors=style.get("edgecolors", "black"),
                       linewidths=style.get("linewidths", 0.8),
                       s=40, zorder=1)

    # --------------------------
    # Animation logic
    # --------------------------
    bee_scatters = []
    plant_scatters = []

    def init():
        nonlocal bee_scatters, plant_scatters
        bee_scatters = []
        plant_scatters = []
        return []

    def update(frame):
        nonlocal bee_scatters, plant_scatters

        # Clear previous visuals
        for scatter in bee_scatters:
            scatter.remove()
        bee_scatters.clear()

        if dynamic_plants:
            for scatter in plant_scatters:
                scatter.remove()
            plant_scatters.clear()

        # ---- Bees
        if frame in bee_df.index.get_level_values("Step"):
            current_bees = bee_df.loc[frame]
            if isinstance(current_bees, pd.Series):
                current_bees = pd.DataFrame([current_bees])
            for _, bee in current_bees.iterrows():
                pos = bee["Bee Positions"]
                style = bee["Plot Code"]
                scat = ax.scatter(pos[0], pos[1],
                                  marker=style.get("marker", "o"),
                                  c=style.get("color"),
                                  edgecolors=style.get("edgecolors", "black"),
                                  linewidths=style.get("linewidths", 0.8),
                                  s=40, zorder=2)
                bee_scatters.append(scat)

        # ---- Plants (if dynamic)
        if dynamic_plants and frame in plant_df.index.get_level_values("Step"):
            current_plants = plant_df.loc[frame]
            if isinstance(current_plants, pd.Series):
                current_plants = pd.DataFrame([current_plants])
            for _, plant in current_plants.iterrows():
                pos = plant["Bee Positions"]
                style = plant["Plot Code"]
                scat = ax.scatter(pos[0], pos[1],
                                  marker=style.get("marker", "o"),
                                  c=style.get("color"),
                                  edgecolors=style.get("edgecolors", "black"),
                                  linewidths=style.get("linewidths", 0.8),
                                  s=40, zorder=1)
                plant_scatters.append(scat)

        ax.set_title(f"Step {frame} - Bee Movement Relative to Plants")
        return bee_scatters + plant_scatters

    # -------------------------- 
    # Create animation
    # --------------------------
    frame_gen = (i for i in tqdm(steps, desc="Rendering animation"))

    anim = FuncAnimation(fig, update, frames=frame_gen, init_func=init,
                         blit=False, interval=300, repeat=False)

    return HTML(anim.to_jshtml())


In [ ]:
visualize_simulation(dynamic_plants=True)